<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m1_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 · Notebook 2 — NumPy for Bioengineers
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

This is a **self-contained** NumPy reference — you do **not** need to consult any other notebook.
It covers everything in Géron's `tools_numpy`, re-ordered around the ideas that matter for machine
learning, **adapted to biomedical data**, and with extras Géron leaves out (a synthetic-ECG signal
thread, real PhysioNet signals, axis-reductions framed as per-feature vs per-sample statistics, and
the NumPy → PyTorch tensor bridge).

Run it in **Google Colab** (*Runtime → Run all*, or **Shift+Enter** per cell). Everything needed for
the core is pre-installed; two optional cells fetch real data and need a network (Colab has one).

**Contents**
1. Why NumPy — the array vs the list
2. Creating arrays & dtypes
3. Shape, axes & reshaping
4. Indexing, slicing, views vs copies
5. Boolean masks & fancy indexing
6. Elementwise maths & universal functions
7. Broadcasting
8. Reductions over an axis (per-feature vs per-sample)
9. Stacking, splitting, concatenating
10. A little linear algebra (full treatment in Module 2)
11. Random numbers & reproducibility
12. Vectorisation & performance
13. Saving & loading
14. **Bio thread:** a synthetic ECG, then a real one from PhysioNet
15. **Extra:** the NumPy → PyTorch tensor bridge

Most sections end with an **Exercise**; run the **Solution** cell to check yourself.

In [ ]:
import numpy as np
print("NumPy", np.__version__)
rng = np.random.default_rng(42)   # modern random generator, seeded for reproducibility

---
## 1 · Why NumPy — the array vs the list

A Python `list` can hold anything, but each element is a separate object and loops run in the slow
interpreter. A **NumPy array** (`ndarray`) is one contiguous block of memory whose elements share a
single numeric type, so whole-array operations run as compiled C loops — typically 10–100× faster and
far more compact. It is *the* unit of scientific computing, and the object Scikit-Learn and PyTorch
expect.

In [ ]:
# Same data, two worlds
heart_rates_list = [60, 72, 80, 68, 75]          # Python list
hr = np.array([60, 72, 80, 68, 75])              # NumPy array
print(type(hr), hr, "| mean:", hr.mean())        # arrays carry maths methods
# A whole-array expression — no loop:
print("z-scored:", (hr - hr.mean()) / hr.std())

---
## 2 · Creating arrays & dtypes

You will create arrays from Python data, from ranges, from constants, and at random. Every array has a
**dtype** (element type); for ML you mostly use `float64`/`float32` and sometimes `uint8` (images) or
`int`.

In [ ]:
a = np.array([1, 2, 3])                 # from a list
b = np.array([[1., 2.], [3., 4.]])     # 2-D from nested lists
z = np.zeros((2, 3))                   # all zeros
o = np.ones(4)                         # all ones
f = np.full((2, 2), 7.0)               # constant
I = np.eye(3)                          # identity matrix
r = np.arange(0, 1, 0.25)              # 0, .25, .5, .75 (stop excluded)
t = np.linspace(0, 1, 5)               # 5 evenly spaced points incl. endpoints
print(a.dtype, b.dtype, z.shape, r, t)

In [ ]:
# dtype matters: images are 8-bit unsigned, models want float32
img = np.array([[0, 128, 255]], dtype=np.uint8)
print("uint8:", img, img.dtype)
print("as float32 in [0,1]:", (img / 255).astype(np.float32))

In [ ]:
# Arrays are fixed-size by design (that's part of why they're fast). np.append / np.delete
# exist, but each one allocates a brand-new array and copies everything into it -- O(n), not
# a real "grow in place" like list.append(). Fine occasionally; a loop of these is a red flag.
grown = np.append(a, [4, 5])
shrunk = np.delete(grown, 1)          # removes the element at index 1
print("grown:", grown, "| shrunk:", shrunk)

---
## 3 · Shape, axes & reshaping

The **shape** is the single most important property of an array — most ML bugs are shape mismatches.
By convention a design matrix is `(n_samples, n_features)`: **rows are samples, columns are features**.

In [ ]:
X = rng.random((6, 3))                 # 6 samples, 3 features
print("shape:", X.shape, "| ndim:", X.ndim, "| size:", X.size, "| dtype:", X.dtype)

v = np.arange(12)
print("reshape to 3x4:\n", v.reshape(3, 4))
print("ravel (flatten):", v.reshape(3, 4).ravel()[:5], "...")
print("transpose of X:", X.T.shape)        # (3, 6)
print("add an axis:", v.reshape(3,4)[:, None, :].shape)  # (3, 1, 4)

In [ ]:
# .reshape() returns a new view without touching the original; you can instead mutate an
# array's shape in place by assigning directly to .shape
v2 = np.arange(12)
v2.shape = (3, 4)             # in-place: v2 itself is now 2-D
print("v2 after in-place reshape:\n", v2)

# Arrays aren't limited to 2-D. Here are 2 stacked 3x4 "frames" -- the kind of shape you'll
# meet again with image batches and channels later in the course
frames = np.arange(24).reshape(2, 3, 4)   # (n_frames, rows, cols)
print("frames shape:", frames.shape, "| ndim:", frames.ndim)
print("first frame:\n", frames[0])
print("element [1, 2, 3]:", frames[1, 2, 3])

---
## 4 · Indexing, slicing, views vs copies

Slicing uses `start:stop:step` (stop excluded, negatives count from the end) and generalises to each
axis as `A[rows, cols]`. **A slice is usually a _view_** onto the same memory, not a copy — writing
through it changes the original. Copy on purpose with `.copy()`.

In [ ]:
A = np.arange(12).reshape(3, 4)
print("A[0, 2] =", A[0, 2])            # single element
print("first row:", A[0])              # A[0, :]
print("last column:", A[:, -1])
print("sub-block A[0:2, 1:3]:\n", A[0:2, 1:3])

In [ ]:
# Views vs copies — the classic 'my data changed by itself' bug
row = A[0]            # a VIEW
row[:] = 0
print("A after writing through the view:\n", A)

A = np.arange(12).reshape(3, 4)
safe = A[0].copy()   # an explicit COPY
safe[:] = 0
print("A unchanged when we use .copy():\n", A)

**Exercise 4.** Given `M = np.arange(20).reshape(4, 5)`: (a) select the third column; (b) select the
bottom-right 2×2 block; (c) reverse the row order. Predict, then run.

In [ ]:
# Solution 4
M = np.arange(20).reshape(4, 5)
print("(a)", M[:, 2])
print("(b)\n", M[-2:, -2:])
print("(c)\n", M[::-1])

---
## 5 · Boolean masks & fancy indexing

A comparison yields a boolean array that can **select** elements — the vectorised replacement for a
loop with an `if`. Integer arrays can select arbitrary elements (**fancy indexing**).

In [ ]:
ages = np.array([34, 71, 19, 58, 65, 22])
mask = ages >= 65
print("mask:", mask, "| seniors:", ages[mask])
ages2 = ages.copy(); ages2[ages2 < 18] = 18        # clip in place via a mask
print("clipped:", ages2)

# One line that IS the ReLU activation:
z = np.array([-2., -0.5, 0., 1.5, 3.])
relu = z.copy(); relu[relu < 0] = 0
print("ReLU:", relu)

# Fancy indexing: pick specific rows
print("rows 0 and 2 of M:\n", M[[0, 2]])

---
## 6 · Elementwise maths & universal functions (ufuncs)

Arithmetic and functions apply **elementwise** across a whole array at compiled speed. NumPy's
elementwise functions (`np.sqrt`, `np.exp`, `np.log`, `np.sin`, …) are called **ufuncs**.

In [ ]:
x = np.linspace(-2, 2, 5)
print("x       :", x)
print("x**2    :", x**2)
print("exp(x)  :", np.round(np.exp(x), 3))
# The logistic (sigmoid) function — used in every classifier:
def sigmoid(z): return 1 / (1 + np.exp(-z))
print("sigmoid :", np.round(sigmoid(x), 3))

---
## 7 · Broadcasting

NumPy combines arrays of **different but compatible** shapes without loops or copies by *broadcasting*
the smaller across the larger. **Rule:** align shapes from the right; each axis must be equal or one of
them 1 (a length-1 axis is stretched). This is how you centre a data matrix or scale image channels.

In [ ]:
X = rng.normal(size=(100, 3))          # 100 samples, 3 features
mu = X.mean(axis=0)                     # shape (3,)
Xc = X - mu                            # (100,3) - (3,) -> (100,3)  [centring]
print("means after centring (~0):", np.round(Xc.mean(axis=0), 6))

# Standardise = centre then scale, all by broadcasting (what StandardScaler does):
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
print("stds after scaling (~1):", np.round(Xs.std(axis=0), 3))

**Exercise 7.** Can shapes `(8, 1, 6)` and `(7, 1)` broadcast? If so, give the result shape.
Then scale every **row** of `X` to unit length (L2 norm = 1) in one line. *Hint:* `keepdims=True`.

In [ ]:
# Solution 7
# (8,1,6) & (7,1) -> pad to (1,7,1): axis -1: 6 vs 1 -> 6; -2: 1 vs 7 -> 7; -3: 8 vs 1 -> 8
print("broadcast result shape -> (8, 7, 6)")
norms = np.linalg.norm(X, axis=1, keepdims=True)   # (100,1)
Xr = X / norms
print("row norms after scaling (~1):", np.round(np.linalg.norm(Xr, axis=1)[:3], 4))

---
## 8 · Reductions over an axis (per-feature vs per-sample)

`sum`, `mean`, `std`, `min`, `max`, `argmax`, `cumsum`… collapse an array. **The axis you name is the
axis that disappears.** For `X` of shape `(n_samples, n_features)`:
`X.mean(axis=0)` → one mean **per feature**; `X.mean(axis=1)` → one mean **per sample**.

In [ ]:
X = rng.random((5, 3))
print("per-feature mean (axis=0):", np.round(X.mean(axis=0), 3), "-> shape", X.mean(axis=0).shape)
print("per-sample  mean (axis=1):", np.round(X.mean(axis=1), 3), "-> shape", X.mean(axis=1).shape)
print("global max:", X.max(), "| index of max feature per sample:", X.argmax(axis=1))
print("cumulative sum down column 0:", np.round(np.cumsum(X[:, 0]), 3))

---
## 9 · Stacking, splitting, concatenating

Assemble and cut arrays along an axis. Common in ML: stacking feature columns, concatenating batches,
splitting a dataset.

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
print("vstack (rows):\n", np.vstack([a, b]))
print("hstack (cols):\n", np.hstack([a, b]))
print("concatenate axis=0 shape:", np.concatenate([a, b], axis=0).shape)
left, right = np.hsplit(np.hstack([a, b]), 2)
print("split back, left:\n", left)

---
## 10 · A little linear algebra (the full treatment is Module 2)

Matrix multiplication is `@` (or `np.matmul`); `*` is **elementwise**, not matrix product — a frequent
beginner trap. A neural-network layer is exactly `X @ W + b`.

In [ ]:
W = rng.normal(size=(3, 2))            # weights: 3 features -> 2 outputs
Xb = rng.normal(size=(4, 3))           # 4 samples, 3 features
bias = np.array([0.1, -0.2])
out = Xb @ W + bias                    # (4,3)@(3,2) -> (4,2), bias broadcasts
print("layer output shape:", out.shape)
print("dot product of two vectors:", np.dot([1,2,3], [4,5,6]))
print("elementwise * is NOT matrix product:", (np.array([1,2]) * np.array([3,4])))

---
## 11 · Random numbers & reproducibility

Use a **seeded generator** (`np.random.default_rng(seed)`) so results are reproducible — essential for
science and for debugging. Draw from named distributions for simulations and synthetic data.

In [ ]:
g = np.random.default_rng(0)
print("uniform [0,1):", np.round(g.random(3), 3))
print("normal(mu=120, sd=15) systolic BP:", np.round(g.normal(120, 15, 4), 1))
print("integers 1..6:", g.integers(1, 7, 5))
print("shuffle a sample order:", g.permutation(np.arange(6)))

---
## 12 · Vectorisation & performance

If you are writing a Python `for` loop over array elements, there is almost always a vectorised version
that is shorter **and** much faster. Here is the difference, measured.

In [ ]:
import time
v = rng.random(2_000_000)

t0 = time.perf_counter()
acc = 0.0
for value in v:           # slow: Python-level loop
    acc += value * value
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
acc_vec = np.sum(v * v)   # fast: one vectorised expression
t_vec = time.perf_counter() - t0

print(f"loop: {t_loop:.3f}s   vectorised: {t_vec:.4f}s   speed-up ~{t_loop/max(t_vec,1e-9):.0f}x")
print("same answer:", np.isclose(acc, acc_vec))

---
## 13 · Saving & loading

Persist arrays with `np.save`/`np.load` (`.npy`), or many with `np.savez`. Text/CSV via
`np.savetxt`/`np.loadtxt` (but Pandas is usually better for labelled tables — Notebook 3).

In [ ]:
arr = rng.random((3, 4))
np.save("demo.npy", arr)
back = np.load("demo.npy")
print("round-trip equal:", np.allclose(arr, back))

---
## 14 · Bio thread — a synthetic ECG, then a real one

NumPy shines on **physiological signals**, which are just 1-D arrays of samples in time. We first build a
synthetic ECG with pure NumPy (always works offline), exercise array skills on it, then show how to pull
a **real** ECG from **PhysioNet** (MIT Laboratory for Computational Physiology).

In [ ]:
# Build a synthetic ECG: P, QRS and T waves as Gaussians, repeated at a heart rate
def gaussian(t, center, width, amp):
    return amp * np.exp(-0.5 * ((t - center) / width) ** 2)

fs = 250                                  # sampling rate (Hz)
dur = 6.0                                 # seconds
t = np.arange(0, dur, 1/fs)               # time axis (a 1-D array)
hr = 72                                   # beats per minute
beat = 60 / hr                            # seconds per beat

ecg = np.zeros_like(t)
for k in range(int(dur / beat) + 1):
    c = k * beat
    ecg += gaussian(t, c - 0.12, 0.025, 0.12)   # P wave
    ecg += gaussian(t, c,         0.012, 1.00)   # R peak (QRS)
    ecg += gaussian(t, c - 0.02,  0.012, -0.18)  # Q
    ecg += gaussian(t, c + 0.02,  0.012, -0.25)  # S
    ecg += gaussian(t, c + 0.18,  0.040, 0.30)   # T wave
ecg += rng.normal(0, 0.02, size=t.shape)         # measurement noise
print("ECG signal:", ecg.shape, "samples,", dur, "s @", fs, "Hz")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(t, ecg, lw=0.9)
plt.xlabel("time (s)"); plt.ylabel("amplitude (mV)"); plt.title("Synthetic ECG")
plt.xlim(0, 3); plt.show()

In [ ]:
# Array skills on a real signal: normalise, smooth (moving average), detect R-peaks — all vectorised
ecg_n = (ecg - ecg.mean()) / ecg.std()            # z-score normalisation (broadcasting)

# Moving-average smoothing via convolution (a vectorised sliding window):
win = 5
smooth = np.convolve(ecg_n, np.ones(win)/win, mode="same")

# Crude R-peak detection: points above a threshold that are local maxima (pure boolean/array logic)
thr = 3.0
above = ecg_n > thr
is_peak = above & (ecg_n > np.roll(ecg_n, 1)) & (ecg_n > np.roll(ecg_n, -1))
peak_times = t[is_peak]
print("detected", peak_times.size, "R-peaks")
if peak_times.size > 1:
    rr = np.diff(peak_times)                      # R-R intervals (s)
    print("mean heart rate from R-R:", round(60 / rr.mean(), 1), "bpm  (target was 72)")

**Optional — real PhysioNet data (needs a network; works in Colab).** Uncomment to stream a real
record from the MIT-BIH Arrhythmia Database. It is wrapped in `try/except` so the notebook never breaks
offline.

In [ ]:
try:
    import wfdb                                   # Colab: pip install wfdb  (often already present)
    rec = wfdb.rdrecord('100', pn_dir='mitdb', sampto=2500)   # 10 s @ 250 Hz from PhysioNet
    real = rec.p_signal[:, 0]                     # first channel -> a NumPy array
    print("real ECG pulled from PhysioNet:", real.shape, rec.units, "| fs =", rec.fs)
except Exception as e:
    print("Skipped real-data fetch (no network or wfdb not installed):", type(e).__name__)
    print("This runs in Colab after:  !pip install wfdb")

**Exercise 14.** Using only NumPy on the synthetic `ecg`, compute: (a) the signal's peak-to-peak
amplitude; (b) its root-mean-square (RMS); (c) the fraction of samples whose absolute value exceeds 0.5.

In [ ]:
# Solution 14
print("(a) peak-to-peak:", round(ecg.max() - ecg.min(), 3))
print("(b) RMS         :", round(np.sqrt(np.mean(ecg**2)), 3))
print("(c) frac > 0.5  :", round(np.mean(np.abs(ecg) > 0.5), 3))

---
## 15 · Extra — the NumPy → PyTorch tensor bridge

The deep-learning half of Géron's book uses **PyTorch tensors**: a tensor is a NumPy array that can also
live on a **GPU** and carry **gradients** (autograd — Module 3). Everything about shape, dtype, indexing
and broadcasting carries over unchanged.

In [ ]:
try:
    import torch
    X32 = rng.random((100, 3)).astype("float32")
    t_ = torch.from_numpy(X32)        # array -> tensor (shares memory on CPU)
    print("tensor:", tuple(t_.shape), t_.dtype, "| back to NumPy:", t_.numpy().shape)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device available:", dev, "(Runtime -> Change runtime type -> GPU for cuda)")
except ImportError:
    print("torch runs in Colab (pre-installed). Same mental model: shape, dtype, broadcasting.")

---
### You now know NumPy
You can create and reshape arrays, index and mask them, broadcast, reduce over the right axis, do matrix
maths, and process a real biomedical signal — the whole toolkit Géron's chapters assume. **Next:**
Notebook 3 (Pandas) puts labels on these arrays for real datasets; Module 2 develops the linear algebra
behind `X @ W`; Module 3 turns the tensor's autograd into gradient descent.